### ЗАДАЧА: Реестр товаров 

В этой задаче нужно собрать простой каталог товаров для внутренней системы закупок.
Данные приходят как строки, часть строк может быть некорректной.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Класс `Product`:
   - поля `id`, `name`, `_price`, `category`
   - `@property price` (только чтение)
   - метод `set_price(value) -> bool`:
     - возвращает `True`, если цена валидна и обновлена
     - возвращает `False`, если цена невалидна
   - `@classmethod from_line(cls, raw)`:
     - если строка корректная, вернуть `(product, "")`
     - если ошибка, вернуть `(None, "текст причины")`.

2. Класс `ProductRegistry`:
   - хранит список товаров
   - метод `add(product) -> (bool, str)`:
     - `(True, "")`, если успешно
     - `(False, "...")`, если `id` уже существует
   - метод `count()`
   - метод `all_products()`
   - метод `has_id(id)`
   - метод `by_category()` -> `{category: count}`
   - метод `avg_price()`.

3. Загрузка данных:
   - пройти по `lines`
   - использовать `Product.from_line(...)`
   - если товар создан, пытаться `registry.add(...)`
   - все проблемные строки складывать в `problems` как `(line, reason)`.

4. Вывод:
   - число валидных товаров
   - статистика по категориям
   - средняя цена
   - список проблемных строк.


- Для проверки дублей хранить set из `id`.
- В `avg_price()` вернуть `0`, если товаров нет.
- Для проверки, что строка число, можно использовать вспомогательную функцию с `str.replace('.', '', 1).isdigit()`.


In [ ]:
lines = [
    "P-1;Mouse;25;Periphery",
    "P-2;Keyboard;45;Periphery",
    "P-3;Cloud VM;120;Services",
    "P-2;Duplicate;50;Periphery",
    "P-4;Broken;-5;Services",
    "P-5;BadPrice;abc;Services",
]

def is_number(text):
    """Возвращает True, если text можно считать числом (включая дроби и знак), иначе False."""
    try:
        float(text)
        return True
    except ValueError:
        return False


class Product:
    def __init__(self, id, name, price, category):
        self._id = id
        self._name = name
        self._price = None
        self._category = category
        # Устанавливаем цену через метод, чтобы проверить валидность
        if not self.set_price(price):
            raise ValueError(f"Invalid price for product {id}: {price}")

    @property
    def price(self):
        """Getter для цены."""
        return self._price

    def set_price(self, value):
        """Устанавливает цену, если значение валидно. Возвращает True при успехе, иначе False."""
        if is_number(value):
            price_value = float(value)
            if price_value >= 0:
                self._price = price_value
                return True
        return False

    @classmethod
    def from_line(cls, raw):
        """Создаёт объект Product из строки raw. Возвращает (product_or_none, reason)."""
        parts = raw.split(';')
        if len(parts) != 4:
            return None, "Invalid line format"

        product_id, name, price_str, category = parts

        # Проверка ID
        if not product_id.startswith('P-'):
            return None, "Invalid product ID format"

        # Проверка цены
        if not is_number(price_str):
            return None, f"Цена должна быть числом: {price_str}"

        price = float(price_str)
        if price < 0:
            return None, f"Отрицательная цена: {price}"

        try:
            product = cls(product_id, name, price, category)
            return product, None
        except ValueError as e:
            return None, str(e)

class ProductRegistry:
    def __init__(self):
        self._products = []
        self._ids = set()

    def add(self, product):
        """Добавляет продукт в реестр. Возвращает (bool, reason)."""
        if product._id in self._ids:
            return False, f"Дублирование ID: {product._id}"
        self._products.append(product)
        self._ids.add(product._id)
        return True, None

    def count(self):
        """Возвращает количество продуктов в реестре."""
        return len(self._products)

    def all_products(self):
        """Возвращает список всех продуктов."""
        return self._products

    def has_id(self, id):
        """Проверяет, есть ли продукт с указанным ID в реестре."""
        return id in self._ids

    def by_category(self):
        """Возвращает словарь: категория → список продуктов."""
        categories = {}
        for product in self._products:
            if product._category not in categories:
                categories[product._category] = []
            categories[product._category].append(product)
        return categories

    def avg_price(self):
        """Возвращает среднюю цену продуктов в реестре. Если продуктов нет — 0."""
        if not self._products:
            return 0
        total = sum(product.price for product in self._products)
        return total / len(self._products)

# Создаём реестр и список проблем
registry = ProductRegistry()
problems = []

# Обрабатываем строки
for line in lines:
    product, reason = Product.from_line(line)
    if product is not None:
        success, add_reason = registry.add(product)
        if not success:
            problems.append((line, add_reason))
    else:
        problems.append((line, reason))

# Выводим результаты
print(f"Количество продуктов: {registry.count()}")
print("\nПродукты по категориям:")
for category, products in registry.by_category().items():
    print(f"  {category}: {len(products)} шт.")
    for product in products:
        print(f"    - {product._name} (ID: {product._id}, цена: {product.price})")

print(f"\nСредняя цена: {registry.avg_price():.2f}")
print("\nПроблемы при загрузке:")
for line, reason in problems:
    print(f"  Строка: '{line}' — причина: {reason}")